# Chicago Strategic Subject List — Analysis

This is the historical SSL dataset: about 399k rows and 48 raw columns. The cleaning is done in `load_data.py`:
- Columns renamed to snake_case.
- Duplicate score columns (`heat_score` etc.) and the constant `status_i` dropped.
- Y/N flags converted to booleans.
- Age bands made ordered categoricals.
- Readable `race` and `sex` labels added.

**Notes:**
- There's no person ID, and about 78k rows are exact duplicates. Pass `drop_duplicates=True` only if you're sure they're the same records.
- Location fields (`latitude`, `community_area`, `census_tract`) are missing for about 43% of rows.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from load_data import load_ssl

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)

df = load_ssl()
df.shape

(398684, 46)

## Overview

In [2]:
df.head()

,ssl_score,predictor_rat_age_at_latest_arrest,predictor_rat_victim_shooting_incidents,predictor_rat_victim_battery_or_assault,predictor_rat_arrests_violent_offenses,predictor_rat_gang_affiliation,predictor_rat_narcotic_arrests,predictor_rat_trend_in_criminal_activity,predictor_rat_uuw_arrests,sex_code_cd,race_code_cd,weapon_i,drug_i,age_group,age_to,stop_order_no,parolee_i,latest_date,latest_dist,majority_dist,dlst,latest_dist_res,weapons_arr_cnt,latest_weapon_arr_date,narcotics_arr_cnt,latest_narcotic_arr_date,idoc_res_city,idoc_res_state_code,idoc_res_zip_code,idoc_cpd_dist,cpd_arrest_i,domestic_arr_cnt,latest_domestic_arr_date,age_curr,ssl_last_ptv_date,trap_status,raw_ssl_score,trap_flags,ssl_flags,latitude,longitude,census_tract,community_area,location,race,sex
0,306,20-30,0,0,0,0,0,-0.3,0,M,WWH,False,False,20-30,20-30,NaN,False,2012,31,NaN,NaN,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,20-30,NaN,NaN,0.09110,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,White Hispanic,Male
1,352,20-30,0,1,0,0,0,0.0,0,M,BLK,False,False,20-30,20-30,NaN,False,2015,31,6.0,NaN,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,20-30,NaN,NaN,0.09949,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,Black,Male
2,349,20-30,0,0,2,0,2,0.0,0,M,BLK,False,False,20-30,20-30,NaN,False,2016,31,NaN,NaN,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,20-30,NaN,NaN,0.09908,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,Black,Male
3,344,20-30,0,0,3,0,0,-0.3,0,M,WHI,False,False,20-30,20-30,NaN,False,2014,31,NaN,NaN,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,20-30,NaN,NaN,0.09810,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,White,Male
4,343,20-30,0,0,1,0,0,0.4,0,M,BLK,False,False,20-30,20-30,NaN,False,2015,31,NaN,NaN,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,20-30,NaN,NaN,0.09794,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,Black,Male


In [3]:
missing = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
missing[missing > 0]

trap_flags                  100.0
ssl_flags                    99.9
stop_order_no                99.6
trap_status                  99.2
idoc_cpd_dist                97.2
ssl_last_ptv_date            96.4
latest_weapon_arr_date       95.2
weapons_arr_cnt              95.2
idoc_res_zip_code            85.4
idoc_res_state_code          85.2
dlst                         85.2
idoc_res_city                85.2
domestic_arr_cnt             82.9
latest_domestic_arr_date     82.9
narcotics_arr_cnt            76.7
latest_narcotic_arr_date     76.7
latitude                     43.8
longitude                    43.8
location                     43.8
community_area               43.7
census_tract                 42.9
majority_dist                36.6
age_curr                      0.1
dtype: float64

In [4]:
df['ssl_score'].describe()

count    398684.000000
mean        279.839743
std          57.996284
min          10.000000
25%         242.000000
50%         289.000000
75%         322.000000
max         500.000000
Name: ssl_score, dtype: float64

## SSL score distribution

In [5]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df['ssl_score'], bins=60, ax=ax)
ax.set(title='SSL score distribution', xlabel='SSL score (0–500)')
plt.show()

/var/folders/sw/z1p4fl791_vbf9w_k5_dmrr40000gn/T/ipykernel_31652/2767598558.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Scores by demographic group

In [6]:
for col in ['race', 'sex', 'age_curr']:
    display(df.groupby(col, observed=True)['ssl_score'].agg(['count', 'mean', 'median']).round(1))

,count,mean,median
race,,,
White Hispanic,99710,283.3,289.0
Black,207842,282.4,293.0
White,82366,269.7,283.0
Asian/Pacific Islander,4617,269.4,283.0
Unknown,1899,276.7,289.0
Black Hispanic,1974,285.4,293.0
American Indian/Alaska Native,276,268.3,276.5


,count,mean,median
sex,,,
Male,302320,278.7,288.0
Female,96307,283.5,295.0
Unknown,57,266.2,281.0


,count,mean,median
age_curr,,,
less than 20,38329,356.6,350.0
20-30,142720,320.9,318.0
30-40,98975,277.4,277.0
40-50,63217,233.3,233.0
50-60,41602,190.4,191.0
60-70,12014,148.7,151.0
70-80,1586,102.0,104.0


In [7]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=df, x='age_curr', y='ssl_score', ax=ax, showfliers=False)
ax.set(title='SSL score by current age band', xlabel='', ylabel='SSL score')
plt.show()

/var/folders/sw/z1p4fl791_vbf9w_k5_dmrr40000gn/T/ipykernel_31652/729986864.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Predictor relationships

In [8]:
pred_cols = [c for c in df.columns if c.startswith('predictor_rat') and df[c].dtype != 'category']
corr = df[pred_cols + ['ssl_score']].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, ax=ax)
plt.show()

/var/folders/sw/z1p4fl791_vbf9w_k5_dmrr40000gn/T/ipykernel_31652/933747336.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
